In [2]:
# ============================================================
# TRACE THE ACE
# 00 — ENVIRONMENT AND PATHS
# CELL 1 — PROJECT DISCOVERY
# ============================================================

from pathlib import Path
from datetime import datetime
import hashlib
import json
import platform
import sys

import numpy as np
import pandas as pd


print("=" * 80)
print("TRACE THE ACE — ENVIRONMENT & PATH SETUP")
print("=" * 80)

print(f"Python : {sys.version}")
print(f"OS     : {platform.system()} {platform.release()}")
print(f"CWD    : {Path.cwd().resolve()}")


# ------------------------------------------------------------
# Resolve project root
# ------------------------------------------------------------

CWD = Path.cwd().resolve()

if CWD.name.lower() == "notebooks":
    PROJECT_ROOT = CWD.parent

else:
    PROJECT_ROOT = None

    for candidate in [CWD, *CWD.parents]:
        if (candidate / "Dataset").is_dir() and (candidate / "Notebooks").is_dir():
            PROJECT_ROOT = candidate
            break

    if PROJECT_ROOT is None:
        raise RuntimeError(
            "Could not determine project root.\n"
            f"Current directory: {CWD}\n"
            "Expected a directory containing Dataset/ and Notebooks/."
        )


DATA_ROOT = PROJECT_ROOT / "Dataset"
TRANSCRIPT_ROOT = DATA_ROOT / "train_transcripts"
NOTEBOOK_ROOT = PROJECT_ROOT / "Notebooks"

TRAIN_FEATURES_PATH = DATA_ROOT / "train_features_TMQTWbS.csv"
TRAIN_LABELS_PATH = DATA_ROOT / "train_labels_44umj2.csv"


SCRATCH_OUTPUT_ROOT = PROJECT_ROOT / "scratch_mastery_outputs"

SETUP_OUTPUT_DIR = (
    SCRATCH_OUTPUT_ROOT / "00_project_setup"
)

PHASE1_ROOT = (
    SCRATCH_OUTPUT_ROOT / "01_data_foundation"
)

FROZEN_FOLD_PATH = (
    SETUP_OUTPUT_DIR / "frozen_fold_manifest.parquet"
)

PATH_REGISTRY_PATH = (
    SETUP_OUTPUT_DIR / "path_registry.json"
)

SETUP_SUMMARY_PATH = (
    SETUP_OUTPUT_DIR / "setup_summary.json"
)

SOURCE_FINGERPRINTS_PATH = (
    SETUP_OUTPUT_DIR / "source_fingerprints.json"
)


# ------------------------------------------------------------
# Create output directories
# ------------------------------------------------------------

SETUP_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PHASE1_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

print("\nResolved project:")
print(f"  PROJECT_ROOT        : {PROJECT_ROOT}")
print(f"  DATA_ROOT           : {DATA_ROOT}")
print(f"  TRANSCRIPT_ROOT     : {TRANSCRIPT_ROOT}")
print(f"  NOTEBOOK_ROOT       : {NOTEBOOK_ROOT}")
print(f"  SCRATCH_OUTPUT_ROOT : {SCRATCH_OUTPUT_ROOT}")

print("\nAuthoritative input files:")
print(f"  TRAIN_FEATURES      : {TRAIN_FEATURES_PATH}")
print(f"  TRAIN_LABELS        : {TRAIN_LABELS_PATH}")

print("\nSetup outputs:")
print(f"  PATH_REGISTRY       : {PATH_REGISTRY_PATH}")
print(f"  SETUP_SUMMARY       : {SETUP_SUMMARY_PATH}")
print(f"  SOURCE_FINGERPRINTS : {SOURCE_FINGERPRINTS_PATH}")
print(f"  FROZEN_FOLD         : {FROZEN_FOLD_PATH}")

TRACE THE ACE — ENVIRONMENT & PATH SETUP
Python : 3.10.20 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:13:20) [MSC v.1942 64 bit (AMD64)]
OS     : Windows 10
CWD    : D:\Competition\Trace-the-race-local\Notebooks

Resolved project:
  PROJECT_ROOT        : D:\Competition\Trace-the-race-local
  DATA_ROOT           : D:\Competition\Trace-the-race-local\Dataset
  TRANSCRIPT_ROOT     : D:\Competition\Trace-the-race-local\Dataset\train_transcripts
  NOTEBOOK_ROOT       : D:\Competition\Trace-the-race-local\Notebooks
  SCRATCH_OUTPUT_ROOT : D:\Competition\Trace-the-race-local\scratch_mastery_outputs

Authoritative input files:
  TRAIN_FEATURES      : D:\Competition\Trace-the-race-local\Dataset\train_features_TMQTWbS.csv
  TRAIN_LABELS        : D:\Competition\Trace-the-race-local\Dataset\train_labels_44umj2.csv

Setup outputs:
  PATH_REGISTRY       : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\00_project_setup\path_registry.json
  SETUP_SUMMARY       : D:\Competition\T

In [3]:
# ============================================================
# CELL 2 — AUTHORITATIVE INPUT DISCOVERY + SCHEMA PREVIEW
# ============================================================

from pathlib import Path
import pandas as pd


print("=" * 80)
print("CELL 2 — AUTHORITATIVE INPUT DISCOVERY")
print("=" * 80)


# ------------------------------------------------------------
# 1. List every CSV directly under Dataset/
# ------------------------------------------------------------

dataset_csvs = sorted(
    [
        p for p in DATA_ROOT.glob("*.csv")
        if p.is_file()
    ],
    key=lambda p: p.name.lower(),
)

print("\nCSV files directly under Dataset/:")
print("-" * 80)

if not dataset_csvs:
    print("NO CSV FILES FOUND")
else:
    for p in dataset_csvs:
        print(
            f"{p.name:<45} "
            f"{p.stat().st_size / (1024**2):>10.2f} MB"
        )


# ------------------------------------------------------------
# 2. Identify feature / label files by filename pattern
# ------------------------------------------------------------

feature_candidates = [
    p for p in dataset_csvs
    if "train_features" in p.name.lower()
]

label_candidates = [
    p for p in dataset_csvs
    if "train_labels" in p.name.lower()
]


print("\nFeature candidates:")
for p in feature_candidates:
    print("  ", p)

print("\nLabel candidates:")
for p in label_candidates:
    print("  ", p)


# ------------------------------------------------------------
# 3. Hard cardinality check
# ------------------------------------------------------------

if len(feature_candidates) != 1:
    raise RuntimeError(
        "Expected exactly ONE train_features*.csv file.\n"
        f"Found {len(feature_candidates)}:\n"
        + "\n".join(
            f"  - {p}"
            for p in feature_candidates
        )
    )

if len(label_candidates) != 1:
    raise RuntimeError(
        "Expected exactly ONE train_labels*.csv file.\n"
        f"Found {len(label_candidates)}:\n"
        + "\n".join(
            f"  - {p}"
            for p in label_candidates
        )
    )


# ------------------------------------------------------------
# 4. Resolve actual authoritative paths
# ------------------------------------------------------------

TRAIN_FEATURES_PATH = feature_candidates[0]
TRAIN_LABELS_PATH = label_candidates[0]


print("\nResolved authoritative inputs:")
print(
    "  TRAIN_FEATURES_PATH:",
    TRAIN_FEATURES_PATH
)

print(
    "  TRAIN_LABELS_PATH  :",
    TRAIN_LABELS_PATH
)


# ------------------------------------------------------------
# 5. Load only the headers first
# ------------------------------------------------------------

features_header = pd.read_csv(
    TRAIN_FEATURES_PATH,
    nrows=0,
)

labels_header = pd.read_csv(
    TRAIN_LABELS_PATH,
    nrows=0,
)


print("\nTrain features columns:")
print(features_header.columns.tolist())

print("\nTrain labels columns:")
print(labels_header.columns.tolist())


# ------------------------------------------------------------
# 6. Validate source contract
# ------------------------------------------------------------

required_feature_columns = {
    "response_id",
    "session_id",
    "learning_objective_id",
    "learning_objective",
}

required_label_columns = {
    "response_id",
    "is_correct",
}


missing_features = (
    required_feature_columns
    - set(features_header.columns)
)

missing_labels = (
    required_label_columns
    - set(labels_header.columns)
)


if missing_features:
    raise AssertionError(
        "Train features schema mismatch.\n"
        f"Missing: {sorted(missing_features)}\n"
        f"Actual: {features_header.columns.tolist()}"
    )


if missing_labels:
    raise AssertionError(
        "Train labels schema mismatch.\n"
        f"Missing: {sorted(missing_labels)}\n"
        f"Actual: {labels_header.columns.tolist()}"
    )


# ------------------------------------------------------------
# 7. Update registry variables in memory
# ------------------------------------------------------------

# IMPORTANT:
# Cell 1 created these variables using hardcoded expected names.
# We now replace them with the actual discovered paths.

print("\n" + "=" * 80)
print("CELL 2 STATUS: PASS")
print("=" * 80)

print("Authoritative feature file:")
print(f"  {TRAIN_FEATURES_PATH}")

print("Authoritative label file:")
print(f"  {TRAIN_LABELS_PATH}")

print("\nFeature schema: PASS")
print("Label schema  : PASS")

CELL 2 — AUTHORITATIVE INPUT DISCOVERY

CSV files directly under Dataset/:
--------------------------------------------------------------------------------
train_features_TMQTWsB.csv                          2.26 MB
train_labels_44ujmj2.csv                            0.40 MB

Feature candidates:
   D:\Competition\Trace-the-race-local\Dataset\train_features_TMQTWsB.csv

Label candidates:
   D:\Competition\Trace-the-race-local\Dataset\train_labels_44ujmj2.csv

Resolved authoritative inputs:
  TRAIN_FEATURES_PATH: D:\Competition\Trace-the-race-local\Dataset\train_features_TMQTWsB.csv
  TRAIN_LABELS_PATH  : D:\Competition\Trace-the-race-local\Dataset\train_labels_44ujmj2.csv

Train features columns:
['response_id', 'session_id', 'learning_objective_id', 'learning_objective']

Train labels columns:
['response_id', 'is_correct']

CELL 2 STATUS: PASS
Authoritative feature file:
  D:\Competition\Trace-the-race-local\Dataset\train_features_TMQTWsB.csv
Authoritative label file:
  D:\Competition\

In [4]:
# ============================================================
# CELL 3 — FEATURE / LABEL FULL DATA VALIDATION
# ============================================================

print("=" * 80)
print("CELL 3 — FEATURE / LABEL FULL DATA VALIDATION")
print("=" * 80)

# ------------------------------------------------------------
# 1. Load authoritative files
# ------------------------------------------------------------

train_features = pd.read_csv(
    TRAIN_FEATURES_PATH
)

train_labels = pd.read_csv(
    TRAIN_LABELS_PATH
)

print("\nLoaded:")
print(
    f"  train_features : {train_features.shape}"
)
print(
    f"  train_labels   : {train_labels.shape}"
)


# ------------------------------------------------------------
# 2. Required schema
# ------------------------------------------------------------

required_feature_columns = {
    "response_id",
    "session_id",
    "learning_objective_id",
    "learning_objective",
}

required_label_columns = {
    "response_id",
    "is_correct",
}

assert required_feature_columns <= set(
    train_features.columns
)

assert required_label_columns <= set(
    train_labels.columns
)


# ------------------------------------------------------------
# 3. Unique response IDs
# ------------------------------------------------------------

assert train_features["response_id"].notna().all(), (
    "train_features.response_id contains nulls."
)

assert train_labels["response_id"].notna().all(), (
    "train_labels.response_id contains nulls."
)

assert train_features["response_id"].is_unique, (
    "train_features.response_id is not unique."
)

assert train_labels["response_id"].is_unique, (
    "train_labels.response_id is not unique."
)


# ------------------------------------------------------------
# 4. Session integrity
# ------------------------------------------------------------

assert train_features["session_id"].notna().all(), (
    "session_id contains nulls."
)

assert (
    train_features["session_id"]
    .astype(str)
    .str.strip()
    .ne("")
    .all()
), (
    "session_id contains empty strings."
)


# ------------------------------------------------------------
# 5. Objective integrity
# ------------------------------------------------------------

assert train_features[
    "learning_objective_id"
].notna().all(), (
    "learning_objective_id contains nulls."
)

assert train_features[
    "learning_objective"
].notna().all(), (
    "learning_objective contains nulls."
)

assert (
    train_features["learning_objective"]
    .astype(str)
    .str.strip()
    .ne("")
    .all()
), (
    "learning_objective contains empty strings."
)


# ------------------------------------------------------------
# 6. Label integrity
# ------------------------------------------------------------

assert train_labels[
    "is_correct"
].notna().all(), (
    "is_correct contains nulls."
)

target_values = set(
    train_labels["is_correct"].unique()
)

print("\nTarget values:")
print(sorted(target_values))

assert target_values <= {0, 1}, (
    f"Unexpected target values: {target_values}"
)


# ------------------------------------------------------------
# 7. Feature / label population alignment
# ------------------------------------------------------------

feature_ids = set(
    train_features["response_id"]
)

label_ids = set(
    train_labels["response_id"]
)

feature_only = feature_ids - label_ids
label_only = label_ids - feature_ids

print("\nResponse population:")
print(
    f"  Feature IDs : {len(feature_ids):,}"
)
print(
    f"  Label IDs   : {len(label_ids):,}"
)
print(
    f"  Feature-only: {len(feature_only):,}"
)
print(
    f"  Label-only  : {len(label_only):,}"
)

assert not feature_only, (
    "Some feature response_ids have no label."
)

assert not label_only, (
    "Some label response_ids have no feature."
)


# ------------------------------------------------------------
# 8. Response/session/objective statistics
# ------------------------------------------------------------

summary = pd.DataFrame({
    "metric": [
        "Response rows",
        "Unique response IDs",
        "Unique sessions",
        "Unique learning objective IDs",
        "Unique objective texts",
        "Positive labels",
        "Negative labels",
        "Positive rate",
    ],
    "value": [
        len(train_features),
        train_features["response_id"].nunique(),
        train_features["session_id"].nunique(),
        train_features["learning_objective_id"].nunique(),
        train_features["learning_objective"].nunique(),
        int(
            (train_labels["is_correct"] == 1).sum()
        ),
        int(
            (train_labels["is_correct"] == 0).sum()
        ),
        round(
            float(
                train_labels["is_correct"].mean()
            ),
            6,
        ),
    ],
})

display(summary)


# ------------------------------------------------------------
# 9. Objective ID → objective text consistency
# ------------------------------------------------------------

objective_consistency = (
    train_features
    .groupby("learning_objective_id")[
        "learning_objective"
    ]
    .nunique()
)

bad_objectives = objective_consistency[
    objective_consistency > 1
]

print("\nObjective ID → text consistency:")
print(
    f"  Total objective IDs : "
    f"{len(objective_consistency):,}"
)

print(
    f"  Inconsistent IDs    : "
    f"{len(bad_objectives):,}"
)

assert len(bad_objectives) == 0, (
    "A learning_objective_id maps to multiple "
    "learning_objective texts."
)


# ------------------------------------------------------------
# 10. Final gate
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CELL 3 STATUS: PASS")
print("=" * 80)

CELL 3 — FEATURE / LABEL FULL DATA VALIDATION

Loaded:
  train_features : (35072, 4)
  train_labels   : (35072, 2)

Target values:
[np.float64(0.0), np.float64(1.0)]

Response population:
  Feature IDs : 35,072
  Label IDs   : 35,072
  Feature-only: 0
  Label-only  : 0


,metric,value
0,Response rows,35072.000000
1,Unique response IDs,35072.000000
2,Unique sessions,22821.000000
3,Unique learning objective IDs,398.000000
4,Unique objective texts,398.000000
5,Positive labels,24637.000000
6,Negative labels,10435.000000
7,Positive rate,0.702469



Objective ID → text consistency:
  Total objective IDs : 398
  Inconsistent IDs    : 0

CELL 3 STATUS: PASS


In [5]:
# ============================================================
# CELL 4 — TRANSCRIPT SCHEMA SMOKE TEST
# ============================================================

print("=" * 80)
print("CELL 4 — TRANSCRIPT SCHEMA SMOKE TEST")
print("=" * 80)

# ------------------------------------------------------------
# 1. Discover transcript CSVs
# ------------------------------------------------------------

transcript_files = sorted(
    [
        p
        for p in TRANSCRIPT_ROOT.rglob("*.csv")
        if p.is_file()
    ],
    key=lambda p: p.relative_to(TRANSCRIPT_ROOT).as_posix(),
)

print(f"\nTranscript files discovered: {len(transcript_files):,}")

assert len(transcript_files) > 0, (
    f"No transcript CSV files found under:\n"
    f"{TRANSCRIPT_ROOT}"
)


# ------------------------------------------------------------
# 2. Required transcript schema
# ------------------------------------------------------------

REQUIRED_TRANSCRIPT_COLUMNS = {
    "session_id",
    "utterance_id",
    "role",
    "content",
    "timestamp",
}


# ------------------------------------------------------------
# 3. Sample files
# ------------------------------------------------------------

# We deliberately inspect only a sample here.
# Full parsing belongs to 02_turn_parser.

SAMPLE_SIZE = min(
    10,
    len(transcript_files),
)

sample_files = transcript_files[:SAMPLE_SIZE]


audit_rows = []


for path in sample_files:

    try:
        sample_df = pd.read_csv(
            path,
            nrows=5,
            dtype=str,
            keep_default_na=False,
        )

        actual_columns = set(
            sample_df.columns
        )

        missing_columns = sorted(
            REQUIRED_TRANSCRIPT_COLUMNS
            - actual_columns
        )

        audit_rows.append({
            "file": path.name,
            "relative_path": (
                path
                .relative_to(TRANSCRIPT_ROOT)
                .as_posix()
            ),
            "rows_sampled": len(sample_df),
            "column_count": len(
                sample_df.columns
            ),
            "missing_required_columns": (
                missing_columns
            ),
            "passed": (
                len(missing_columns) == 0
            ),
        })

    except Exception as exc:

        audit_rows.append({
            "file": path.name,
            "relative_path": (
                path
                .relative_to(TRANSCRIPT_ROOT)
                .as_posix()
            ),
            "rows_sampled": 0,
            "column_count": 0,
            "missing_required_columns": [
                f"READ_ERROR: {exc}"
            ],
            "passed": False,
        })


transcript_schema_audit = pd.DataFrame(
    audit_rows
)

display(
    transcript_schema_audit
)


# ------------------------------------------------------------
# 4. Hard gate
# ------------------------------------------------------------

failed = transcript_schema_audit[
    ~transcript_schema_audit["passed"]
]

if not failed.empty:

    print("\nFAILED FILES:")
    display(failed)

    raise AssertionError(
        "Transcript schema smoke test failed."
    )


# ------------------------------------------------------------
# 5. Show one representative transcript
# ------------------------------------------------------------

representative_path = sample_files[0]

representative_df = pd.read_csv(
    representative_path,
    nrows=5,
    dtype=str,
    keep_default_na=False,
)

print("\n" + "=" * 80)
print("REPRESENTATIVE TRANSCRIPT")
print("=" * 80)

print(
    f"File: {representative_path.name}"
)

display(
    representative_df
)


# ------------------------------------------------------------
# 6. Final status
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CELL 4 STATUS: PASS")
print("=" * 80)

CELL 4 — TRANSCRIPT SCHEMA SMOKE TEST

Transcript files discovered: 22,821


,file,relative_path,rows_sampled,column_count,missing_required_columns,passed
0,aaaedit.csv,aaaedit.csv,5,5,[],True
1,aaaptjd.csv,aaaptjd.csv,5,5,[],True
2,aabkeov.csv,aabkeov.csv,5,5,[],True
3,aacggvb.csv,aacggvb.csv,5,5,[],True
4,aadexbc.csv,aadexbc.csv,5,5,[],True
5,aadinwu.csv,aadinwu.csv,5,5,[],True
6,aadljmq.csv,aadljmq.csv,5,5,[],True
7,aadmino.csv,aadmino.csv,5,5,[],True
8,aadsgow.csv,aadsgow.csv,5,5,[],True
9,aadylxv.csv,aadylxv.csv,5,5,[],True



REPRESENTATIVE TRANSCRIPT
File: aaaedit.csv


,session_id,utterance_id,role,content,timestamp
0,aaaedit,0,tutor,Hello?,00:00:00
1,aaaedit,1,background,[unclear],00:00:00
2,aaaedit,2,background,Hello.,00:00:01
3,aaaedit,3,tutor,"Hi, Lachlan. How are you doing today?",00:00:01
4,aaaedit,4,student,Good.,00:00:07



CELL 4 STATUS: PASS


In [6]:
# ============================================================
# CELL 5 — SOURCE FINGERPRINTING
# ============================================================

import hashlib
import json
from datetime import datetime


print("=" * 80)
print("CELL 5 — SOURCE FINGERPRINTING")
print("=" * 80)


# ------------------------------------------------------------
# 1. SHA-256 helper
# ------------------------------------------------------------

def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:

    digest = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


# ------------------------------------------------------------
# 2. Verify transcript inventory
# ------------------------------------------------------------

transcript_files = sorted(
    [
        p
        for p in TRANSCRIPT_ROOT.rglob("*.csv")
        if p.is_file()
    ],
    key=lambda p: p.relative_to(
        TRANSCRIPT_ROOT
    ).as_posix(),
)

assert len(transcript_files) == 22_821, (
    "Transcript file count changed after Cell 4.\n"
    f"Expected: 22,821\n"
    f"Found   : {len(transcript_files):,}"
)

print(
    f"\nTranscript files to fingerprint: "
    f"{len(transcript_files):,}"
)


# ------------------------------------------------------------
# 3. Fingerprint train feature / label files
# ------------------------------------------------------------

print("\nHashing feature file...")

TRAIN_FEATURES_SHA256 = sha256_file(
    TRAIN_FEATURES_PATH
)

print(
    "train_features SHA256:",
    TRAIN_FEATURES_SHA256
)


print("\nHashing label file...")

TRAIN_LABELS_SHA256 = sha256_file(
    TRAIN_LABELS_PATH
)

print(
    "train_labels SHA256:",
    TRAIN_LABELS_SHA256
)


# ------------------------------------------------------------
# 4. Fingerprint every transcript
# ------------------------------------------------------------

print("\nHashing transcript files...")
print(
    "This may take some time because "
    "there are 22,821 files."
)


transcript_fingerprint_rows = []


for i, path in enumerate(
    transcript_files,
    start=1,
):

    relative_path = (
        path
        .relative_to(TRANSCRIPT_ROOT)
        .as_posix()
    )

    file_hash = sha256_file(path)

    transcript_fingerprint_rows.append({
        "relative_path": relative_path,
        "size_bytes": path.stat().st_size,
        "sha256": file_hash,
    })

    if i % 1000 == 0:
        print(
            f"  Fingerprinted "
            f"{i:,} / {len(transcript_files):,}"
        )


transcript_file_manifest = pd.DataFrame(
    transcript_fingerprint_rows
)


# ------------------------------------------------------------
# 5. Validate fingerprint manifest
# ------------------------------------------------------------

assert len(
    transcript_file_manifest
) == len(transcript_files)

assert transcript_file_manifest[
    "relative_path"
].is_unique

assert transcript_file_manifest[
    "sha256"
].notna().all()

assert transcript_file_manifest[
    "sha256"
].str.len().eq(64).all()


# ------------------------------------------------------------
# 6. Build deterministic directory fingerprint
# ------------------------------------------------------------

directory_digest = hashlib.sha256()

for row in transcript_file_manifest.itertuples(
    index=False
):

    canonical_line = (
        f"{row.relative_path}\t"
        f"{row.size_bytes}\t"
        f"{row.sha256}\n"
    )

    directory_digest.update(
        canonical_line.encode("utf-8")
    )


TRANSCRIPT_DIRECTORY_SHA256 = (
    directory_digest.hexdigest()
)


print(
    "\nTranscript directory SHA256:",
    TRANSCRIPT_DIRECTORY_SHA256
)


# ------------------------------------------------------------
# 7. Save transcript file manifest
# ------------------------------------------------------------

transcript_manifest_path = (
    SETUP_OUTPUT_DIR
    / "transcript_file_manifest.parquet"
)


transcript_file_manifest.to_parquet(
    transcript_manifest_path,
    index=False,
)


print(
    "\nSaved transcript manifest:"
)

print(
    f"  {transcript_manifest_path}"
)


# ------------------------------------------------------------
# 8. Save source fingerprints
# ------------------------------------------------------------

source_fingerprints = {
    "created_at": (
        datetime.now()
        .astimezone()
        .isoformat()
    ),

    "train_features": {
        "path": str(
            TRAIN_FEATURES_PATH
        ),
        "sha256": (
            TRAIN_FEATURES_SHA256
        ),
        "size_bytes": (
            TRAIN_FEATURES_PATH.stat()
            .st_size
        ),
    },

    "train_labels": {
        "path": str(
            TRAIN_LABELS_PATH
        ),
        "sha256": (
            TRAIN_LABELS_SHA256
        ),
        "size_bytes": (
            TRAIN_LABELS_PATH.stat()
            .st_size
        ),
    },

    "transcripts": {
        "root": str(
            TRANSCRIPT_ROOT
        ),
        "file_count": int(
            len(transcript_files)
        ),
        "directory_sha256": (
            TRANSCRIPT_DIRECTORY_SHA256
        ),
    },
}


with open(
    SOURCE_FINGERPRINTS_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        source_fingerprints,
        f,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )


print(
    "\nSaved source fingerprints:"
)

print(
    f"  {SOURCE_FINGERPRINTS_PATH}"
)


# ------------------------------------------------------------
# 9. Preview manifest
# ------------------------------------------------------------

print("\nTranscript fingerprint manifest preview:")

display(
    transcript_file_manifest.head(10)
)


# ------------------------------------------------------------
# 10. Final gate
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CELL 5 STATUS: PASS")
print("=" * 80)

print(
    f"Transcript files fingerprinted: "
    f"{len(transcript_file_manifest):,}"
)

print(
    f"Transcript directory hash: "
    f"{TRANSCRIPT_DIRECTORY_SHA256}"
)

CELL 5 — SOURCE FINGERPRINTING

Transcript files to fingerprint: 22,821

Hashing feature file...
train_features SHA256: 71bea3abb76a1cff5e1eaa75b9cbcfaf26d0419f6274b83a199ed520047a5063

Hashing label file...
train_labels SHA256: d98ee4389e5cde3f66d6d15b7b574261024a80e405958eca333d3c1921fd65b9

Hashing transcript files...
This may take some time because there are 22,821 files.
  Fingerprinted 1,000 / 22,821
  Fingerprinted 2,000 / 22,821
  Fingerprinted 3,000 / 22,821
  Fingerprinted 4,000 / 22,821
  Fingerprinted 5,000 / 22,821
  Fingerprinted 6,000 / 22,821
  Fingerprinted 7,000 / 22,821
  Fingerprinted 8,000 / 22,821
  Fingerprinted 9,000 / 22,821
  Fingerprinted 10,000 / 22,821
  Fingerprinted 11,000 / 22,821
  Fingerprinted 12,000 / 22,821
  Fingerprinted 13,000 / 22,821
  Fingerprinted 14,000 / 22,821
  Fingerprinted 15,000 / 22,821
  Fingerprinted 16,000 / 22,821
  Fingerprinted 17,000 / 22,821
  Fingerprinted 18,000 / 22,821
  Fingerprinted 19,000 / 22,821
  Fingerprinted 20,000

,relative_path,size_bytes,sha256
0,aaaedit.csv,23586,eed066ebe1c1ec2b68d4c8c1bc4afd080063f78c495d37...
1,aaaptjd.csv,42935,ce0834d60e9a31e7402cd97a3e3bd69300d6d94370bc89...
2,aabkeov.csv,24232,e01c76f619e3a1684131b1f177f7a6fb0e24d89b3b24ff...
3,aacggvb.csv,28249,ad8392f205308a0228fa4d197cadad093550519416a497...
4,aadexbc.csv,16780,81fa67086e99339eb81dbbf14e9f275d92eb4c3da6de92...
5,aadinwu.csv,18763,459036f4c5534596ac9e13fc3d02464a41c7fbef7a1972...
6,aadljmq.csv,31280,8d1cfdd5b5111f17ff8e4c459c0c671745fd9af69e66cf...
7,aadmino.csv,20993,3f4837ef5dda21a6284c03d609af84ce90bc99f5e64217...
8,aadsgow.csv,25661,0599083394bf183ef9b0d6d2e23234dfe63851fd950922...
9,aadylxv.csv,29167,737b5733623674ae2ac066394eeaafa876ac8097b7b232...



CELL 5 STATUS: PASS
Transcript files fingerprinted: 22,821
Transcript directory hash: 3f563b9911cd2f2e4d01dd5a457509ceaac6eb31ae880148034d7b2ca89402df


In [7]:
# ============================================================
# CELL 6 — FROZEN FIVE-FOLD SESSION-GROUPED MANIFEST
# ============================================================

from sklearn.model_selection import StratifiedGroupKFold
import numpy as np


print("=" * 80)
print("CELL 6 — FROZEN FIVE-FOLD SESSION-GROUPED MANIFEST")
print("=" * 80)


# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

N_SPLITS = 5

# One-time seed.
# Once the manifest is created, this seed must NOT be changed.
FOLD_RANDOM_STATE = 20260812


# ------------------------------------------------------------
# 1. Build fold source
# ------------------------------------------------------------

fold_source = (
    train_features[
        [
            "response_id",
            "session_id",
        ]
    ]
    .merge(
        train_labels[
            [
                "response_id",
                "is_correct",
            ]
        ],
        on="response_id",
        how="inner",
        validate="one_to_one",
    )
)


print("\nFold source:")
print(
    f"  Rows    : {len(fold_source):,}"
)

print(
    f"  Sessions: "
    f"{fold_source['session_id'].nunique():,}"
)


# ------------------------------------------------------------
# 2. Hard input checks
# ------------------------------------------------------------

assert len(fold_source) == len(train_features)

assert fold_source["response_id"].is_unique

assert fold_source["session_id"].notna().all()

assert fold_source["is_correct"].isin(
    [0, 1]
).all()


# ------------------------------------------------------------
# 3. Check for existing frozen manifest
# ------------------------------------------------------------

FROZEN_FOLD_PATH = (
    SETUP_OUTPUT_DIR
    / "frozen_fold_manifest.parquet"
)


if FROZEN_FOLD_PATH.exists():

    # ========================================================
    # EXISTING MANIFEST — NEVER REGENERATE
    # ========================================================

    print("\nExisting frozen fold manifest found.")
    print(
        f"Path: {FROZEN_FOLD_PATH}"
    )

    frozen_folds = pd.read_parquet(
        FROZEN_FOLD_PATH
    )

    print(
        f"Rows: {len(frozen_folds):,}"
    )

    required_columns = {
        "response_id",
        "session_id",
        "fold",
    }

    missing_columns = (
        required_columns
        - set(frozen_folds.columns)
    )

    assert not missing_columns, (
        "Existing frozen fold manifest is missing: "
        f"{sorted(missing_columns)}"
    )

    print(
        "Existing manifest will NOT be regenerated."
    )


else:

    # ========================================================
    # FIRST CREATION
    # ========================================================

    print("\nNo frozen fold manifest found.")

    print(
        "Creating one-time session-grouped "
        "5-fold split..."
    )

    print(
        f"Random state: {FOLD_RANDOM_STATE}"
    )


    splitter = StratifiedGroupKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=FOLD_RANDOM_STATE,
    )


    fold_source = fold_source.copy()

    fold_source["fold"] = -1


    X_dummy = np.zeros(
        len(fold_source),
        dtype=np.int8,
    )

    y = (
        fold_source["is_correct"]
        .astype(int)
    )

    groups = (
        fold_source["session_id"]
        .astype(str)
    )


    # --------------------------------------------------------
    # Generate folds
    # --------------------------------------------------------

    for fold_id, (_, valid_idx) in enumerate(
        splitter.split(
            X_dummy,
            y,
            groups,
        )
    ):

        fold_source.loc[
            valid_idx,
            "fold"
        ] = fold_id


    # --------------------------------------------------------
    # Hard assignment checks
    # --------------------------------------------------------

    assert fold_source[
        "fold"
    ].isin(
        range(N_SPLITS)
    ).all()


    assert not (
        fold_source["fold"] == -1
    ).any()


    # --------------------------------------------------------
    # Create final manifest
    # --------------------------------------------------------

    frozen_folds = (
        fold_source[
            [
                "response_id",
                "session_id",
                "fold",
            ]
        ]
        .sort_values(
            "response_id"
        )
        .reset_index(
            drop=True
        )
    )


    # --------------------------------------------------------
    # Response uniqueness
    # --------------------------------------------------------

    assert frozen_folds[
        "response_id"
    ].is_unique


    # --------------------------------------------------------
    # Session leakage check
    # --------------------------------------------------------

    session_fold_counts = (
        frozen_folds
        .groupby(
            "session_id"
        )["fold"]
        .nunique()
    )


    assert (
        session_fold_counts <= 1
    ).all(), (
        "SESSION LEAKAGE DETECTED: "
        "at least one session appears in "
        "multiple folds."
    )


    # --------------------------------------------------------
    # All five folds must exist
    # --------------------------------------------------------

    observed_folds = set(
        frozen_folds[
            "fold"
        ].unique()
    )

    assert observed_folds == {
        0, 1, 2, 3, 4
    }, (
        f"Expected folds {{0,1,2,3,4}}, "
        f"found {observed_folds}"
    )


    # --------------------------------------------------------
    # Save ONCE
    # --------------------------------------------------------

    frozen_folds.to_parquet(
        FROZEN_FOLD_PATH,
        index=False,
    )


    print(
        "\nFrozen fold manifest CREATED:"
    )

    print(
        f"  {FROZEN_FOLD_PATH}"
    )


# ------------------------------------------------------------
# 4. Final manifest validation
# ------------------------------------------------------------

assert len(
    frozen_folds
) == len(train_features)


assert set(
    frozen_folds[
        "response_id"
    ]
) == set(
    train_features[
        "response_id"
    ]
)


assert frozen_folds[
    "response_id"
].is_unique


session_fold_counts = (
    frozen_folds
    .groupby(
        "session_id"
    )["fold"]
    .nunique()
)


assert (
    session_fold_counts <= 1
).all()


assert set(
    frozen_folds[
        "fold"
    ].unique()
) == {
    0, 1, 2, 3, 4
}


# ------------------------------------------------------------
# 5. Fold statistics
# ------------------------------------------------------------

fold_summary = (
    frozen_folds
    .merge(
        train_labels[
            [
                "response_id",
                "is_correct",
            ]
        ],
        on="response_id",
        how="left",
        validate="one_to_one",
    )
    .groupby(
        "fold"
    )
    .agg(
        responses=(
            "response_id",
            "size",
        ),

        sessions=(
            "session_id",
            "nunique",
        ),

        positives=(
            "is_correct",
            "sum",
        ),

        positive_rate=(
            "is_correct",
            "mean",
        ),
    )
    .reset_index()
)


fold_summary[
    "positive_rate"
] = (
    fold_summary[
        "positive_rate"
    ]
    .round(6)
)


print("\nFold summary:")
display(fold_summary)


# ------------------------------------------------------------
# 6. Session leakage audit
# ------------------------------------------------------------

leakage_audit = (
    frozen_folds
    .groupby("session_id")
    .agg(
        fold_count=(
            "fold",
            "nunique",
        )
    )
    .reset_index()
)


print("\nSession leakage audit:")

print(
    "  Sessions checked:",
    len(leakage_audit)
)

print(
    "  Max fold count:",
    leakage_audit[
        "fold_count"
    ].max()
)


assert (
    leakage_audit[
        "fold_count"
    ].max()
    == 1
)


# ------------------------------------------------------------
# 7. Final status
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CELL 6 STATUS: PASS")
print("=" * 80)

print(
    f"Frozen manifest rows : "
    f"{len(frozen_folds):,}"
)

print(
    f"Folds                 : "
    f"{sorted(frozen_folds['fold'].unique())}"
)

print(
    "Session leakage       : NONE"
)

print(
    "Manifest immutable    : YES"
)

CELL 6 — FROZEN FIVE-FOLD SESSION-GROUPED MANIFEST

Fold source:
  Rows    : 35,072
  Sessions: 22,821

Existing frozen fold manifest found.
Path: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\00_project_setup\frozen_fold_manifest.parquet
Rows: 35,072
Existing manifest will NOT be regenerated.

Fold summary:


,fold,responses,sessions,positives,positive_rate
0,0,6958,4539,4879.0,0.701207
1,1,7050,4557,4958.0,0.703262
2,2,7023,4560,4915.0,0.699843
3,3,7081,4594,4978.0,0.703008
4,4,6960,4571,4907.0,0.705029



Session leakage audit:
  Sessions checked: 22821
  Max fold count: 1

CELL 6 STATUS: PASS
Frozen manifest rows : 35,072
Folds                 : [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
Session leakage       : NONE
Manifest immutable    : YES


In [8]:
# ============================================================
# CELL 6B — FROZEN FOLD FINAL VERIFICATION
# ============================================================

print("=" * 80)
print("CELL 6B — FROZEN FOLD FINAL VERIFICATION")
print("=" * 80)

FROZEN_FOLD_PATH = (
    SETUP_OUTPUT_DIR
    / "frozen_fold_manifest.parquet"
)

assert FROZEN_FOLD_PATH.exists(), (
    f"Frozen fold manifest not found:\n{FROZEN_FOLD_PATH}"
)

frozen_folds_check = pd.read_parquet(
    FROZEN_FOLD_PATH
)

print("\nManifest:")
print(f"  Path : {FROZEN_FOLD_PATH}")
print(f"  Rows : {len(frozen_folds_check):,}")

print("\nColumns:")
print(frozen_folds_check.columns.tolist())


# ------------------------------------------------------------
# Required schema
# ------------------------------------------------------------

required_columns = {
    "response_id",
    "session_id",
    "fold",
}

assert required_columns <= set(
    frozen_folds_check.columns
)


# ------------------------------------------------------------
# Response coverage
# ------------------------------------------------------------

assert frozen_folds_check[
    "response_id"
].notna().all()

assert frozen_folds_check[
    "response_id"
].is_unique

assert len(
    frozen_folds_check
) == len(
    train_features
)

assert set(
    frozen_folds_check["response_id"]
) == set(
    train_features["response_id"]
)


# ------------------------------------------------------------
# Fold validity
# ------------------------------------------------------------

observed_folds = set(
    frozen_folds_check["fold"].unique()
)

print(
    "\nObserved folds:",
    sorted(observed_folds)
)

assert observed_folds == {
    0, 1, 2, 3, 4
}


# ------------------------------------------------------------
# Session leakage
# ------------------------------------------------------------

session_fold_counts = (
    frozen_folds_check
    .groupby("session_id")["fold"]
    .nunique()
)

max_fold_count = int(
    session_fold_counts.max()
)

multi_fold_sessions = (
    session_fold_counts[
        session_fold_counts > 1
    ]
)

print(
    "\nSession leakage audit:"
)

print(
    f"  Sessions checked : "
    f"{len(session_fold_counts):,}"
)

print(
    f"  Max fold count   : "
    f"{max_fold_count}"
)

print(
    f"  Leaking sessions : "
    f"{len(multi_fold_sessions):,}"
)

assert max_fold_count == 1, (
    "SESSION LEAKAGE DETECTED."
)


# ------------------------------------------------------------
# Fold summary
# ------------------------------------------------------------

fold_summary = (
    frozen_folds_check
    .merge(
        train_labels[
            ["response_id", "is_correct"]
        ],
        on="response_id",
        how="left",
        validate="one_to_one",
    )
    .groupby("fold")
    .agg(
        responses=("response_id", "size"),
        sessions=("session_id", "nunique"),
        positives=("is_correct", "sum"),
        positive_rate=("is_correct", "mean"),
    )
    .reset_index()
)

fold_summary["positive_rate"] = (
    fold_summary["positive_rate"]
    .round(6)
)

display(fold_summary)


# ------------------------------------------------------------
# Final gate
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("CELL 6B STATUS: PASS")
print("=" * 80)

print("✓ 35,072 responses covered")
print("✓ 22,821 sessions covered")
print("✓ Five folds present")
print("✓ Response IDs unique")
print("✓ No session crosses folds")
print("✓ Frozen manifest verified")

CELL 6B — FROZEN FOLD FINAL VERIFICATION

Manifest:
  Path : D:\Competition\Trace-the-race-local\scratch_mastery_outputs\00_project_setup\frozen_fold_manifest.parquet
  Rows : 35,072

Columns:
['response_id', 'session_id', 'fold']

Observed folds: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

Session leakage audit:
  Sessions checked : 22,821
  Max fold count   : 1
  Leaking sessions : 0


,fold,responses,sessions,positives,positive_rate
0,0,6958,4539,4879.0,0.701207
1,1,7050,4557,4958.0,0.703262
2,2,7023,4560,4915.0,0.699843
3,3,7081,4594,4978.0,0.703008
4,4,6960,4571,4907.0,0.705029



CELL 6B STATUS: PASS
✓ 35,072 responses covered
✓ 22,821 sessions covered
✓ Five folds present
✓ Response IDs unique
✓ No session crosses folds
✓ Frozen manifest verified


In [16]:
# ============================================================
# CELL 7 — SETUP SUMMARY
# ============================================================

from datetime import datetime
import json
import sys
from pathlib import Path

import pandas as pd


print("=" * 80)
print("CELL 7 — SETUP SUMMARY")
print("=" * 80)


# ------------------------------------------------------------
# 1. Load authoritative source tables
# ------------------------------------------------------------

assert TRAIN_FEATURES_PATH.is_file()
assert TRAIN_LABELS_PATH.is_file()
assert FROZEN_FOLD_PATH.is_file()
assert TRANSCRIPT_ROOT.is_dir()


train_features = pd.read_csv(
    TRAIN_FEATURES_PATH
)

train_labels = pd.read_csv(
    TRAIN_LABELS_PATH
)


# ------------------------------------------------------------
# 2. Data paths
# ------------------------------------------------------------

required_paths = {
    "project_root": PROJECT_ROOT,
    "data_root": DATA_ROOT,
    "notebook_root": NOTEBOOK_ROOT,
    "transcript_root": TRANSCRIPT_ROOT,
    "train_features": TRAIN_FEATURES_PATH,
    "train_labels": TRAIN_LABELS_PATH,
    "frozen_fold": FROZEN_FOLD_PATH,
    "scratch_output_root": SCRATCH_OUTPUT_ROOT,
}

data_paths_ready = all(
    Path(p).exists()
    for p in required_paths.values()
)


# ------------------------------------------------------------
# 3. Feature / label schema
# ------------------------------------------------------------

REQUIRED_FEATURE_COLUMNS = {
    "response_id",
    "session_id",
    "learning_objective_id",
    "learning_objective",
}

REQUIRED_LABEL_COLUMNS = {
    "response_id",
    "is_correct",
}

feature_label_schema_ready = (
    REQUIRED_FEATURE_COLUMNS
    <= set(train_features.columns)
    and
    REQUIRED_LABEL_COLUMNS
    <= set(train_labels.columns)
    and
    train_features["response_id"].notna().all()
    and
    train_labels["response_id"].notna().all()
    and
    train_features["response_id"].is_unique
    and
    train_labels["response_id"].is_unique
    and
    set(train_features["response_id"])
    ==
    set(train_labels["response_id"])
)


# ------------------------------------------------------------
# 4. Frozen folds
# ------------------------------------------------------------

frozen_folds_check = pd.read_parquet(
    FROZEN_FOLD_PATH
)

required_fold_columns = {
    "response_id",
    "session_id",
    "fold",
}

fold_schema_ok = (
    required_fold_columns
    <= set(frozen_folds_check.columns)
)

fold_response_coverage_ok = (
    len(frozen_folds_check)
    ==
    len(train_features)
    and
    frozen_folds_check["response_id"].is_unique
    and
    set(frozen_folds_check["response_id"])
    ==
    set(train_features["response_id"])
)

fold_values_ok = (
    set(
        frozen_folds_check[
            "fold"
        ].dropna().unique()
    )
    ==
    {0, 1, 2, 3, 4}
)

session_fold_counts = (
    frozen_folds_check
    .groupby("session_id")["fold"]
    .nunique()
)

no_session_leakage = (
    len(session_fold_counts) > 0
    and
    session_fold_counts.max() == 1
)

frozen_folds_ready = (
    fold_schema_ok
    and
    fold_response_coverage_ok
    and
    fold_values_ok
    and
    no_session_leakage
)


# ------------------------------------------------------------
# 5. Transcript source
# ------------------------------------------------------------

TRANSCRIPT_SOURCE_TYPE = "external_path"

transcript_files = sorted(
    [
        p
        for p in TRANSCRIPT_ROOT.rglob("*.csv")
        if p.is_file()
    ],
    key=lambda p: (
        p.relative_to(
            TRANSCRIPT_ROOT
        ).as_posix()
    ),
)

transcript_source_ready = (
    TRANSCRIPT_ROOT.is_dir()
    and
    len(transcript_files) > 0
)


# ------------------------------------------------------------
# 6. Git safety — deterministic check
# ------------------------------------------------------------

GITIGNORE_PATH = (
    PROJECT_ROOT / ".gitignore"
)

GITIGNORE_RULE = (
    "scratch_mastery_outputs/"
)


scratch_output_git_safe = False

if GITIGNORE_PATH.is_file():

    gitignore_text = (
        GITIGNORE_PATH
        .read_text(
            encoding="utf-8",
            errors="ignore",
        )
    )

    normalized_gitignore = (
        gitignore_text
        .replace("\\", "/")
    )

    scratch_output_git_safe = (
        GITIGNORE_RULE
        in normalized_gitignore
    )


# HARD verification:
# If the rule exists in the actual root .gitignore,
# the flag MUST be True.

if GITIGNORE_PATH.is_file():

    direct_gitignore_check = (
        GITIGNORE_RULE
        in GITIGNORE_PATH.read_text(
            encoding="utf-8",
            errors="ignore",
        ).replace("\\", "/")
    )

    assert (
        scratch_output_git_safe
        ==
        direct_gitignore_check
    )


# ------------------------------------------------------------
# 7. Environment flags
# ------------------------------------------------------------

python_312_ready = (
    sys.version_info[:2]
    == (3, 12)
)

cuda_ready = False


# ------------------------------------------------------------
# 8. Dataset counts
# ------------------------------------------------------------

response_count = int(
    len(train_features)
)

label_count = int(
    len(train_labels)
)

session_count = int(
    train_features[
        "session_id"
    ].nunique()
)

objective_count = int(
    train_features[
        "learning_objective_id"
    ].nunique(
        dropna=True
    )
)

positive_rate = float(
    train_labels[
        "is_correct"
    ].mean()
)

frozen_fold_rows = int(
    len(frozen_folds_check)
)


# ------------------------------------------------------------
# 9. Setup summary
# ------------------------------------------------------------

setup_summary = {

    "data_paths_ready":
        bool(data_paths_ready),

    "feature_label_schema_ready":
        bool(feature_label_schema_ready),

    "frozen_folds_ready":
        bool(frozen_folds_ready),

    "transcript_source_ready":
        bool(transcript_source_ready),

    "scratch_output_git_safe":
        bool(scratch_output_git_safe),

    "python_312_ready":
        bool(python_312_ready),

    "cuda_ready":
        bool(cuda_ready),

    "response_count":
        response_count,

    "label_count":
        label_count,

    "session_count":
        session_count,

    "objective_count":
        objective_count,

    "positive_rate":
        positive_rate,

    "frozen_fold_rows":
        frozen_fold_rows,

    "transcript_file_count":
        len(transcript_files),

    "setup_timestamp":
        datetime.now()
        .astimezone()
        .isoformat(),
}


# ------------------------------------------------------------
# 10. Save
# ------------------------------------------------------------

SETUP_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SETUP_SUMMARY_PATH = (
    SETUP_OUTPUT_DIR
    / "setup_summary.json"
)

with open(
    SETUP_SUMMARY_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        setup_summary,
        f,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )


# ------------------------------------------------------------
# 11. READ-BACK VERIFICATION
# ------------------------------------------------------------

with open(
    SETUP_SUMMARY_PATH,
    "r",
    encoding="utf-8",
) as f:

    persisted_setup_summary = json.load(f)


assert (
    persisted_setup_summary[
        "scratch_output_git_safe"
    ]
    is True
), (
    "setup_summary.json was written with "
    "scratch_output_git_safe=False."
)


# ------------------------------------------------------------
# 12. Final display
# ------------------------------------------------------------

display(
    pd.DataFrame(
        [
            {
                "check": key,
                "value": value,
            }
            for key, value
            in setup_summary.items()
        ]
    )
)


print("\n" + "=" * 80)
print("CELL 7 STATUS: PASS")
print("=" * 80)

print(
    "data_paths_ready           =",
    data_paths_ready,
)

print(
    "feature_label_schema_ready =",
    feature_label_schema_ready,
)

print(
    "frozen_folds_ready         =",
    frozen_folds_ready,
)

print(
    "transcript_source_ready    =",
    transcript_source_ready,
)

print(
    "scratch_output_git_safe    =",
    scratch_output_git_safe,
)

print(
    "\nSaved and verified:"
)

print(
    SETUP_SUMMARY_PATH
)

CELL 7 — SETUP SUMMARY


,check,value
0,data_paths_ready,True
1,feature_label_schema_ready,True
2,frozen_folds_ready,True
3,transcript_source_ready,True
4,scratch_output_git_safe,True
5,python_312_ready,False
6,cuda_ready,False
7,response_count,35072
8,label_count,35072
9,session_count,22821



CELL 7 STATUS: PASS
data_paths_ready           = True
feature_label_schema_ready = True
frozen_folds_ready         = True
transcript_source_ready    = True
scratch_output_git_safe    = True

Saved and verified:
D:\Competition\Trace-the-race-local\scratch_mastery_outputs\00_project_setup\setup_summary.json


In [18]:
# ============================================================
# CELL 8 — SOURCE FINGERPRINT MANIFEST
# ============================================================

import hashlib
import json
from pathlib import Path

import pandas as pd


print("=" * 80)
print("CELL 8 — SOURCE FINGERPRINT MANIFEST")
print("=" * 80)


# ------------------------------------------------------------
# 1. SHA-256 helper
# ------------------------------------------------------------

def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:

    digest = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


# ------------------------------------------------------------
# 2. Authoritative file hashes
# ------------------------------------------------------------

TRAIN_FEATURES_SHA256 = sha256_file(
    TRAIN_FEATURES_PATH
)

TRAIN_LABELS_SHA256 = sha256_file(
    TRAIN_LABELS_PATH
)

FROZEN_FOLD_MANIFEST_SHA256 = sha256_file(
    FROZEN_FOLD_PATH
)


# ------------------------------------------------------------
# 3. Transcript manifest from Cell 5 of Phase-0
# ------------------------------------------------------------

TRANSCRIPT_MANIFEST_PATH = (
    SETUP_OUTPUT_DIR
    / "transcript_file_manifest.parquet"
)


assert TRANSCRIPT_MANIFEST_PATH.is_file(), (
    "Transcript fingerprint manifest is missing:\n"
    f"{TRANSCRIPT_MANIFEST_PATH}"
)


transcript_file_manifest = pd.read_parquet(
    TRANSCRIPT_MANIFEST_PATH
)


required_transcript_columns = {
    "relative_path",
    "size_bytes",
    "sha256",
}


assert required_transcript_columns <= set(
    transcript_file_manifest.columns
)


# ------------------------------------------------------------
# 4. Deterministic transcript directory hash
# ------------------------------------------------------------

directory_digest = hashlib.sha256()


manifest_sorted = (
    transcript_file_manifest
    .sort_values(
        "relative_path"
    )
    .reset_index(drop=True)
)


for row in manifest_sorted.itertuples(
    index=False
):

    canonical_line = (
        f"{row.relative_path}\t"
        f"{int(row.size_bytes)}\t"
        f"{row.sha256}\n"
    )

    directory_digest.update(
        canonical_line.encode("utf-8")
    )


TRANSCRIPT_DIRECTORY_SHA256 = (
    directory_digest.hexdigest()
)


# ------------------------------------------------------------
# 5. Source-compatible flat fingerprint contract
# ------------------------------------------------------------

source_fingerprints = {

    "train_features_sha256":
        TRAIN_FEATURES_SHA256,

    "train_labels_sha256":
        TRAIN_LABELS_SHA256,

    "frozen_fold_manifest_sha256":
        FROZEN_FOLD_MANIFEST_SHA256,

    "transcript_directory_sha256":
        TRANSCRIPT_DIRECTORY_SHA256,
}


# ------------------------------------------------------------
# 6. Save
# ------------------------------------------------------------

SOURCE_FINGERPRINTS_PATH = (
    SETUP_OUTPUT_DIR
    / "source_fingerprints.json"
)


with open(
    SOURCE_FINGERPRINTS_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        source_fingerprints,
        f,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )


# ------------------------------------------------------------
# 7. Read-back verification
# ------------------------------------------------------------

with open(
    SOURCE_FINGERPRINTS_PATH,
    "r",
    encoding="utf-8",
) as f:

    verified_fingerprints = json.load(f)


required_keys = {
    "train_features_sha256",
    "train_labels_sha256",
    "frozen_fold_manifest_sha256",
    "transcript_directory_sha256",
}


assert required_keys <= set(
    verified_fingerprints
)


assert all(
    isinstance(
        verified_fingerprints[key],
        str,
    )
    and
    len(
        verified_fingerprints[key]
    ) == 64
    for key in required_keys
)


print("\n" + "=" * 80)
print("CELL 8 STATUS: PASS")
print("=" * 80)

for key in sorted(required_keys):

    print(
        f"{key}: "
        f"{verified_fingerprints[key]}"
    )

CELL 8 — SOURCE FINGERPRINT MANIFEST

CELL 8 STATUS: PASS
frozen_fold_manifest_sha256: 90daffad48b5a267027999d682615bc550a9e8656fe73de04e6625e404b2e3e0
train_features_sha256: 71bea3abb76a1cff5e1eaa75b9cbcfaf26d0419f6274b83a199ed520047a5063
train_labels_sha256: d98ee4389e5cde3f66d6d15b7b574261024a80e405958eca333d3c1921fd65b9
transcript_directory_sha256: 3f563b9911cd2f2e4d01dd5a457509ceaac6eb31ae880148034d7b2ca89402df


In [19]:
# ============================================================
# CELL 9 — FINAL PATH REGISTRY
# ============================================================

import json


print("=" * 80)
print("CELL 9 — FINAL PATH REGISTRY")
print("=" * 80)


path_registry = {

    "project_root":
        str(PROJECT_ROOT),

    "data_root":
        str(DATA_ROOT),

    "notebook_root":
        str(NOTEBOOK_ROOT),

    # Source notebook's expected name
    "advanced_notebook_root":
        str(NOTEBOOK_ROOT),

    "transcript_source":
        str(TRANSCRIPT_ROOT),

    "transcript_source_type":
        "external_path",

    "train_features_path":
        str(TRAIN_FEATURES_PATH),

    "train_labels_path":
        str(TRAIN_LABELS_PATH),

    "frozen_fold_manifest_path":
        str(FROZEN_FOLD_PATH),

    "scratch_output_root":
        str(SCRATCH_OUTPUT_ROOT),

    "setup_output_dir":
        str(SETUP_OUTPUT_DIR),

    "phase1_output_root":
        str(
            SCRATCH_OUTPUT_ROOT
            / "01_data_foundation"
        ),

    "notebook_root":
        str(NOTEBOOK_ROOT),
}


# ------------------------------------------------------------
# Remove accidental duplicate dictionary key issue
# by reconstructing cleanly
# ------------------------------------------------------------

path_registry = {
    "project_root": str(PROJECT_ROOT),
    "data_root": str(DATA_ROOT),
    "notebook_root": str(NOTEBOOK_ROOT),
    "advanced_notebook_root": str(NOTEBOOK_ROOT),
    "transcript_source": str(TRANSCRIPT_ROOT),
    "transcript_source_type": "external_path",
    "train_features_path": str(TRAIN_FEATURES_PATH),
    "train_labels_path": str(TRAIN_LABELS_PATH),
    "frozen_fold_manifest_path": str(FROZEN_FOLD_PATH),
    "scratch_output_root": str(SCRATCH_OUTPUT_ROOT),
    "setup_output_dir": str(SETUP_OUTPUT_DIR),
    "phase1_output_root": str(
        SCRATCH_OUTPUT_ROOT
        / "01_data_foundation"
    ),
}


with open(
    PATH_REGISTRY_PATH,
    "w",
    encoding="utf-8",
) as f:

    json.dump(
        path_registry,
        f,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    )


# ------------------------------------------------------------
# Read-back
# ------------------------------------------------------------

with open(
    PATH_REGISTRY_PATH,
    "r",
    encoding="utf-8",
) as f:

    verified_registry = json.load(f)


required_registry_keys = {
    "project_root",
    "data_root",
    "notebook_root",
    "advanced_notebook_root",
    "transcript_source",
    "transcript_source_type",
    "train_features_path",
    "train_labels_path",
    "frozen_fold_manifest_path",
    "scratch_output_root",
    "setup_output_dir",
    "phase1_output_root",
}


assert required_registry_keys <= set(
    verified_registry
)


print("\n" + "=" * 80)
print("CELL 9 STATUS: PASS")
print("=" * 80)

print(
    f"Saved: {PATH_REGISTRY_PATH}"
)

CELL 9 — FINAL PATH REGISTRY

CELL 9 STATUS: PASS
Saved: D:\Competition\Trace-the-race-local\scratch_mastery_outputs\00_project_setup\path_registry.json


In [20]:
# ============================================================
# CELL 10 — FINAL SETUP HARD GATE
# ============================================================

print("=" * 80)
print("CELL 10 — FINAL SETUP HARD GATE")
print("=" * 80)


# ------------------------------------------------------------
# File existence
# ------------------------------------------------------------

checks = []


def add_check(
    name,
    condition,
):
    checks.append({
        "check": name,
        "passed": bool(condition),
    })


add_check(
    "Project root exists",
    PROJECT_ROOT.is_dir(),
)

add_check(
    "Dataset root exists",
    DATA_ROOT.is_dir(),
)

add_check(
    "Transcript directory exists",
    TRANSCRIPT_ROOT.is_dir(),
)

add_check(
    "Train features exists",
    TRAIN_FEATURES_PATH.is_file(),
)

add_check(
    "Train labels exists",
    TRAIN_LABELS_PATH.is_file(),
)

add_check(
    "Transcript manifest exists",
    (
        SETUP_OUTPUT_DIR
        / "transcript_file_manifest.parquet"
    ).is_file(),
)

add_check(
    "Source fingerprints exists",
    SOURCE_FINGERPRINTS_PATH.is_file(),
)

add_check(
    "Setup summary exists",
    SETUP_SUMMARY_PATH.is_file(),
)

add_check(
    "Frozen fold exists",
    FROZEN_FOLD_PATH.is_file(),
)


# ------------------------------------------------------------
# Dataset checks
# ------------------------------------------------------------

add_check(
    "35,072 responses present",
    len(train_features) == 35_072,
)

add_check(
    "22,821 sessions present",
    train_features[
        "session_id"
    ].nunique() == 22_821,
)

add_check(
    "Feature/label IDs match",
    set(
        train_features[
            "response_id"
        ]
    )
    ==
    set(
        train_labels[
            "response_id"
        ]
    ),
)

add_check(
    "Five folds present",
    set(
        frozen_folds[
            "fold"
        ].unique()
    )
    ==
    {0, 1, 2, 3, 4},
)


# ------------------------------------------------------------
# Leakage check
# ------------------------------------------------------------

session_fold_counts = (
    frozen_folds
    .groupby(
        "session_id"
    )["fold"]
    .nunique()
)

add_check(
    "No session crosses folds",
    session_fold_counts.max() == 1,
)


# ------------------------------------------------------------
# Manifest coverage
# ------------------------------------------------------------

add_check(
    "All responses have fold",
    len(
        frozen_folds
    )
    ==
    len(
        train_features
    ),
)

add_check(
    "Fold response IDs unique",
    frozen_folds[
        "response_id"
    ].is_unique,
)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

setup_gate = pd.DataFrame(
    checks
)

display(setup_gate)


# ------------------------------------------------------------
# Final hard gate
# ------------------------------------------------------------

SETUP_READY = bool(
    setup_gate[
        "passed"
    ].all()
)


print("\n" + "=" * 80)
print(
    f"SETUP_READY = {SETUP_READY}"
)
print("=" * 80)


if not SETUP_READY:

    failed = setup_gate[
        ~setup_gate["passed"]
    ]

    raise AssertionError(
        "SETUP HARD GATE FAILED:\n"
        + failed.to_string(
            index=False
        )
    )


print(
    "\nPHASE-0 SETUP IS FROZEN."
)

print(
    "Do not regenerate the fold manifest."
)

print(
    "Do not modify source fingerprints."
)

print(
    "Do not use test data for fitting."
)

CELL 10 — FINAL SETUP HARD GATE


,check,passed
0,Project root exists,True
1,Dataset root exists,True
2,Transcript directory exists,True
3,Train features exists,True
4,Train labels exists,True
5,Transcript manifest exists,True
6,Source fingerprints exists,True
7,Setup summary exists,True
8,Frozen fold exists,True
9,"35,072 responses present",True



SETUP_READY = True

PHASE-0 SETUP IS FROZEN.
Do not regenerate the fold manifest.
Do not modify source fingerprints.
Do not use test data for fitting.


In [21]:
# ============================================================
# FIX — PROJECT ROOT .gitignore
# ============================================================

from pathlib import Path


PROJECT_ROOT = Path(
    r"D:\Competition\Trace-the-race-local"
)

ROOT_GITIGNORE = PROJECT_ROOT / ".gitignore"

RULE = "scratch_mastery_outputs/"


# ------------------------------------------------------------
# Preserve existing root .gitignore if present
# ------------------------------------------------------------

if ROOT_GITIGNORE.exists():

    text = ROOT_GITIGNORE.read_text(
        encoding="utf-8",
        errors="ignore",
    )

else:

    text = ""


# ------------------------------------------------------------
# Add rule if missing
# ------------------------------------------------------------

normalized_lines = {
    line.strip().replace("\\", "/")
    for line in text.splitlines()
    if line.strip()
}


if RULE not in normalized_lines:

    if text and not text.endswith("\n"):
        text += "\n"

    text += (
        "\n"
        "# Trace The Ace generated artifacts\n"
        "scratch_mastery_outputs/\n"
    )

    ROOT_GITIGNORE.write_text(
        text,
        encoding="utf-8",
    )


# ------------------------------------------------------------
# Verify
# ------------------------------------------------------------

assert ROOT_GITIGNORE.is_file()

verified_text = ROOT_GITIGNORE.read_text(
    encoding="utf-8",
    errors="ignore",
)

verified_lines = {
    line.strip().replace("\\", "/")
    for line in verified_text.splitlines()
    if line.strip()
}

assert RULE in verified_lines


print("=" * 80)
print("PROJECT ROOT GITIGNORE: PASS")
print("=" * 80)

print(
    f"Path : {ROOT_GITIGNORE}"
)

print(
    f"Rule : {RULE}"
)

PROJECT ROOT GITIGNORE: PASS
Path : D:\Competition\Trace-the-race-local\.gitignore
Rule : scratch_mastery_outputs/
